In [1]:
import random
import statistics

In [2]:
class Town:
    def __init__(self, size=40, empty=0.10, threshold=0.40, seed=0):
        self.rng = random.Random(seed)
        self.size, self.threshold = size, threshold

        cells = size * size
        n_empty = round(empty * cells)
        n_each = (cells - n_empty) // 2

        # 0 = empty, 1 and 2 = the two groups
        values = (
            [1] * n_each
            + [2] * (cells - n_empty - n_each)
            + [0] * n_empty
        )

        self.rng.shuffle(values)
        self.grid = [
            values[r * size:(r + 1) * size]
            for r in range(size)
        ]

    def neighbours(self, r, c):
        # The eight surrounding cells; edges wrap around
        s = self.size
        return [
            self.grid[(r + a) % s][(c + b) % s]
            for a in (-1, 0, 1)
            for b in (-1, 0, 1)
            if (a, b) != (0, 0)
        ]

    def agents(self):
        return [
            (r, c)
            for r in range(self.size)
            for c in range(self.size)
            if self.grid[r][c] != 0
        ]

    def empties(self):
        return [
            (r, c)
            for r in range(self.size)
            for c in range(self.size)
            if self.grid[r][c] == 0
        ]

    def like_share(self, r, c):
        # TODO 1: Consider only occupied neighbours
        group = self.grid[r][c]
        occupied = [
            value for value in self.neighbours(r, c)
            if value != 0
        ]

        # If there are no occupied neighbours, the resident is content
        if len(occupied) == 0:
            return 1.0

        # Fraction of occupied neighbours from the same group
        same_group = sum(
            value == group for value in occupied
        )

        return same_group / len(occupied)

    def is_content(self, r, c):
        # TODO 2: Check whether the like share reaches the threshold
        return self.like_share(r, c) >= self.threshold

    def discontent(self):
        return [
            (r, c) for r, c in self.agents()
            if not self.is_content(r, c)
        ]

    def average_like_share(self):
        a = self.agents()
        return sum(
            self.like_share(r, c) for r, c in a
        ) / len(a)

    def play_round(self):
        # TODO 3a: Find unhappy residents and shuffle their order
        movers = self.discontent()
        self.rng.shuffle(movers)

        # TODO 3b: Find all currently empty cells
        empties = self.empties()

        # TODO 3c: Move each unhappy resident to a random empty cell
        for r, c in movers:
            j = self.rng.randrange(len(empties))
            er, ec = empties[j]

            # Transfer the resident to the empty cell
            self.grid[er][ec] = self.grid[r][c]

            # Make the original cell empty
            self.grid[r][c] = 0

            # Update the list of empty cells
            empties[j] = (r, c)

        # TODO 3d: Return the number of residents who moved
        return len(movers)

In [3]:
town = Town(threshold=0.4, seed=3)

print(
    "Start: like share",
    round(town.average_like_share(), 2),
    ", discontent:",
    len(town.discontent())
)

rounds = 0

while True:
    moved = town.play_round()
    rounds += 1

    if rounds in (1, 2, 5, 10, 15) or moved == 0:
        print(
            f"Round {rounds}: moved {moved}, "
            f"like share {town.average_like_share():.2f}"
        )

    if moved == 0:
        break

Start: like share 0.49 , discontent: 459
Round 1: moved 459, like share 0.62
Round 2: moved 245, like share 0.70
Round 5: moved 60, like share 0.81
Round 10: moved 13, like share 0.84
Round 15: moved 3, like share 0.85
Round 22: moved 0, like share 0.85


In [4]:
for threshold in [0.2, 0.4, 0.6]:
    rounds_used, shares = [], []

    for s in range(10):
        town = Town(threshold=threshold, seed=s)
        r = 0

        while r < 100 and town.play_round() > 0:
            r += 1

        rounds_used.append(r)
        shares.append(town.average_like_share())

    print(
        f"threshold {threshold}: "
        f"rounds {statistics.mean(rounds_used):.1f}, "
        f"like share {statistics.mean(shares):.2f}"
    )

threshold 0.2: rounds 6.2, like share 0.58
threshold 0.4: rounds 16.6, like share 0.82
threshold 0.6: rounds 91.7, like share 0.97
